# 04 — Model Evaluation

**AI-Based Retinal Imaging and Ophthalmic Screening System**

This notebook:
1. Loads the trained model
2. Runs inference on the held-out test set
3. Computes all evaluation metrics
4. Plots confusion matrix and per-class breakdowns
5. Demonstrates Grad-CAM on a sample image
6. Discusses medical interpretation

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from config.config import (
    BEST_MODEL_PATH, CLASS_NAMES_PATH, SPLITS_DIR,
    FIGURES_DIR, METRICS_DIR, TARGET_CLASSES
)
from src.utils import load_class_names, ensure_dir, save_json

print('Imports OK')

In [ ]:
# ── 1. Load Model ─────────────────────────────────────────────────────────────
from tensorflow import keras

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f'Model not found: {BEST_MODEL_PATH}\n'
        'Run: python -m src.train  or  notebook 03_model_training.ipynb'
    )

model = keras.models.load_model(str(BEST_MODEL_PATH))
print(f'Model loaded: {BEST_MODEL_PATH}')
model.summary()

In [ ]:
# ── 2. Load class names ───────────────────────────────────────────────────────
try:
    class_names = load_class_names(CLASS_NAMES_PATH)
except FileNotFoundError:
    class_names = TARGET_CLASSES
print(f'Class names: {class_names}')

In [ ]:
# ── 3. Load Test Set ──────────────────────────────────────────────────────────
from datasets import load_dataset
from src.preprocessing import build_numpy_arrays

test_csv = SPLITS_DIR / 'test.csv'
if not test_csv.exists():
    raise FileNotFoundError(
        f'Test CSV not found: {test_csv}\n'
        'Run: python -m src.dataset  then  python -m src.train'
    )

df_test = pd.read_csv(test_csv)
ds = load_dataset('bumbledeep/odir', split='train')

print(f'Test samples: {len(df_test):,}')
print('Building test arrays (this may take a few minutes) ...')
X_test, y_test = build_numpy_arrays(ds, df_test)
print(f'X_test: {X_test.shape}  y_test: {y_test.shape}')

In [ ]:
# ── 4. Predictions ────────────────────────────────────────────────────────────
print('Running predictions ...')
probs  = model.predict(X_test, batch_size=32, verbose=1)
y_pred = np.argmax(probs, axis=1)
print(f'Predictions complete. Shape: {probs.shape}')

In [ ]:
# ── 5. Metrics ────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, f1_score, precision_score, recall_score
)
from sklearn.preprocessing import label_binarize

acc       = accuracy_score(y_test, y_pred)
macro_f1  = f1_score(y_test, y_pred, average='macro', zero_division=0)
macro_pre = precision_score(y_test, y_pred, average='macro', zero_division=0)
macro_rec = recall_score(y_test, y_pred, average='macro', zero_division=0)

n_classes = len(class_names)
y_bin     = label_binarize(y_test, classes=list(range(n_classes)))
try:
    roc_auc = roc_auc_score(y_bin, probs, multi_class='ovr', average='macro')
except Exception as e:
    roc_auc = None
    print(f'ROC-AUC failed: {e}')

print('=' * 50)
print(f'  Accuracy        : {acc*100:.2f}%')
print(f'  Macro F1        : {macro_f1*100:.2f}%')
print(f'  Macro Precision : {macro_pre*100:.2f}%')
print(f'  Macro Recall    : {macro_rec*100:.2f}%')
if roc_auc:
    print(f'  Macro ROC-AUC   : {roc_auc:.4f}')
print('=' * 50)

print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

In [ ]:
# ── 6. Confusion Matrix ───────────────────────────────────────────────────────
ensure_dir(FIGURES_DIR)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5, linecolor='white',
    cbar_kws={'shrink': 0.8}, ax=ax,
)
ax.set_xlabel('Predicted Label', fontsize=12, labelpad=10)
ax.set_ylabel('True Label',      fontsize=12, labelpad=10)
ax.set_title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved.')

In [ ]:
# ── 7. Per-Class Precision, Recall, F1 Bar Plot ───────────────────────────────
from sklearn.metrics import precision_recall_fscore_support

pre, rec, f1, sup = precision_recall_fscore_support(
    y_test, y_pred, labels=list(range(n_classes)), zero_division=0
)

x = np.arange(n_classes)
w = 0.25
palette = ['#2196F3', '#4CAF50', '#FF9800']

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w, pre, w, label='Precision', color=palette[0], edgecolor='white')
ax.bar(x,     rec, w, label='Recall',    color=palette[1], edgecolor='white')
ax.bar(x + w, f1,  w, label='F1-Score',  color=palette[2], edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(class_names, fontsize=11)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_title('Per-Class Evaluation Metrics', fontsize=13, fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'per_class_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8. ROC Curves ─────────────────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#4CAF50', '#F44336', '#2196F3', '#FF9800']

for i, (cls, color) in enumerate(zip(class_names, colors)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
    auc_score   = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{cls} (AUC={auc_score:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curves — One-vs-Rest', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 9. Grad-CAM on Sample Test Images ────────────────────────────────────────
from src.explainability import generate_gradcam_overlay

fig, axes = plt.subplots(len(class_names), 3, figsize=(12, 4 * len(class_names)))

for row, cls_name in enumerate(class_names):
    # Find a correctly predicted test sample for this class
    cls_idx   = class_names.index(cls_name)
    correct   = np.where((y_test == cls_idx) & (y_pred == cls_idx))[0]
    if len(correct) == 0:
        for ax in axes[row]:
            ax.text(0.5, 0.5, 'No correct prediction', ha='center', va='center',
                    transform=ax.transAxes)
            ax.axis('off')
        continue

    sample_idx  = correct[0]
    pil_img_arr = (X_test[sample_idx] * 255).astype(np.uint8)

    from PIL import Image
    pil_img = Image.fromarray(pil_img_arr)
    
    gc_result = generate_gradcam_overlay(model, pil_img, class_index=cls_idx)
    conf      = float(probs[sample_idx, cls_idx])

    titles = ['Original', 'Grad-CAM Heatmap', f'Overlay ({cls_name} — {conf*100:.1f}%)']
    imgs   = [
        gc_result.get('original') or pil_img,
        gc_result.get('heatmap'),
        gc_result.get('overlay') or pil_img,
    ]
    for ax, title, img in zip(axes[row], titles, imgs):
        if img is not None:
            ax.imshow(img)
        ax.set_title(title, fontsize=10)
        ax.axis('off')

plt.suptitle('Grad-CAM Explanations — One Sample per Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'gradcam_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grad-CAM examples saved.')

In [ ]:
# ── 10. Save evaluation metrics JSON ─────────────────────────────────────────
ensure_dir(METRICS_DIR)

from sklearn.metrics import classification_report
report_dict = classification_report(
    y_test, y_pred, target_names=class_names, output_dict=True, zero_division=0
)

eval_metrics = {
    'accuracy':          round(float(acc), 4),
    'macro_f1':          round(float(macro_f1), 4),
    'macro_precision':   round(float(macro_pre), 4),
    'macro_recall':      round(float(macro_rec), 4),
    'roc_auc_macro':     round(float(roc_auc), 4) if roc_auc else None,
    'confusion_matrix':  cm.tolist(),
    'per_class_report':  report_dict,
    'class_names':       class_names,
    'n_test_samples':    int(len(y_test)),
}

save_json(eval_metrics, METRICS_DIR / 'evaluation_metrics.json')
save_json(report_dict,  METRICS_DIR / 'classification_report.json')

print('Metrics saved to reports/metrics/')

## Medical Interpretation Note

- **Recall (Sensitivity)** is the most important metric for screening tasks.
  High recall minimises missed disease cases (false negatives).
- **Accuracy alone** is insufficient — class imbalance can inflate accuracy.
- **ROC-AUC** provides a threshold-independent measure of discriminability.
- **Grad-CAM** must not be interpreted as confirmation of disease.

> ⚠️ All predictions must be reviewed by a qualified ophthalmologist.